In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load the Iris dataset
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Train a Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Check its accuracy
test_predictions = model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, test_predictions))

# Choose the first flower in the test set
x_instance = X_test.iloc[[0]]

predicted_class = model.predict(x_instance)[0]

print("\nFlower to explain:")
print(x_instance)
print("\nPredicted class:", iris.target_names[predicted_class])
print("Actual class:", iris.target_names[y_test.iloc[0]])

Test accuracy: 0.9

Flower to explain:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
38                4.4               3.0                1.3               0.2

Predicted class: setosa
Actual class: setosa


In [ ]:
from sklearn.linear_model import LinearRegression

#1 Perturbation,  Create 5000 slightly changed flowers
np.random.seed(42)
number_of_samples = 5000

feature_std = X_train.std().to_numpy()
original_flower = x_instance.iloc[0].to_numpy()

noise = np.random.normal(
    0,
    0.20 * feature_std,
    size=(number_of_samples, 4)
)

new_flowers = original_flower + noise
new_flowers[0] = original_flower

new_flowers_df = pd.DataFrame(
    new_flowers,
    columns=X_train.columns
)

# 2 Prediction
forest_probabilities = model.predict_proba(
    new_flowers_df
)[:, predicted_class]

# 3 Weighted sensibility, the closer the perturbed instance is the bigger the weight is

# Calculate how far each new flower is from the original
scaled_changes = (new_flowers - original_flower) / feature_std
distances_squared = np.sum(scaled_changes ** 2, axis=1)

# Nearby flowers get larger weights
kernel_width = 0.75 * np.sqrt(4)
weights = np.exp(-distances_squared / kernel_width ** 2)

#Surrogate model
# Train a weighted linear model
manual_model = LinearRegression()
manual_model.fit(
    scaled_changes,
    forest_probabilities,
    sample_weight=weights
)

# Show the feature coefficients
manual_results = pd.DataFrame({
    "Feature": X_train.columns,
    "Manual coefficient": manual_model.coef_
})

print("Class being explained:", iris.target_names[predicted_class])
print("\nManual explanation:")
print(manual_results.to_string(index=False))

print("\nWeighted local R²:", round(
    manual_model.score(
        scaled_changes,
        forest_probabilities,
        sample_weight=weights
    ),
    3
))

Class being explained: setosa

Manual explanation:
          Feature  Manual coefficient
sepal length (cm)            0.000638
 sepal width (cm)            0.001887
petal length (cm)           -0.003417
 petal width (cm)           -0.004491

Weighted local R²: 0.012


In [ ]:
widths = {
    "Small": kernel_width / 3,
    "Medium": kernel_width,
    "Large": kernel_width * 3
}

results = pd.DataFrame(index=X_train.columns)

for name, width in widths.items():
    new_weights = np.exp(-distances_squared / width ** 2)

    local_model = LinearRegression()
    local_model.fit(
        scaled_changes,
        forest_probabilities,
        sample_weight=new_weights
    )

    results[name] = local_model.coef_

    score = local_model.score(
        scaled_changes,
        forest_probabilities,
        sample_weight=new_weights
    )

    print(name, "width:", round(width, 3))
    print("Weighted local R²:", round(score, 3))
    print()

print("Coefficients for each width:")
print(results.round(3))

print("\nFeatures that change sign:")

found_flip = False

for feature in results.index:
    values = results.loc[feature]

    if (values > 0).any() and (values < 0).any():
        print(feature)
        found_flip = True

if not found_flip:
    print("None")

Small width: 0.5
Weighted local R²: 0.005

Medium width: 1.5
Weighted local R²: 0.012

Large width: 4.5
Weighted local R²: 0.014

Coefficients for each width:
                   Small  Medium  Large
sepal length (cm)  0.000   0.001  0.001
sepal width (cm)   0.001   0.002  0.002
petal length (cm) -0.001  -0.003 -0.004
petal width (cm)  -0.001  -0.004 -0.006

Features that change sign:
None


In [ ]:
!pip -q install lime

In [ ]:

from lime.lime_tabular import LimeTabularExplainer

# Give the LIME package information about our training data
explainer = LimeTabularExplainer(
    training_data=X_train.to_numpy(),
    feature_names=list(X_train.columns),
    class_names=list(iris.target_names),
    mode="classification",
    discretize_continuous=False,
    sample_around_instance=True,
    random_state=42
)

# LIME sends NumPy arrays; add column names before prediction
def predict_with_names(samples):
    samples_df = pd.DataFrame(
        samples,
        columns=X_train.columns
    )
    return model.predict_proba(samples_df)

# Explain the same flower as in Problems 1–3
lime_result = explainer.explain_instance(
    data_row=x_instance.iloc[0].to_numpy(),
    predict_fn=predict_with_names,
    labels=(int(predicted_class),),
    num_features=4,
    num_samples=5000
)

print("Class being explained:", iris.target_names[predicted_class])

print("\nLIME package explanation:")
for feature, value in lime_result.as_list(label=int(predicted_class)):
    print(feature, ":", round(value, 4))

print("\nManual explanation from Problem 2:")
print(manual_results.to_string(index=False))

Class being explained: setosa

LIME package explanation:
petal width (cm) : -0.1614
petal length (cm) : -0.1583
sepal length (cm) : -0.0098
sepal width (cm) : 0.001

Manual explanation from Problem 2:
          Feature  Manual coefficient
sepal length (cm)            0.000638
 sepal width (cm)            0.001887
petal length (cm)           -0.003417
 petal width (cm)           -0.004491
